### CI: average tip


In [18]:
import numpy as np 
import pandas as pd 
import scipy.stats as stats 
cols = ["tpep_pickup_datetime", "fare_amount", "tip_amount", "payment_type"]
df = pd.read_parquet('yellow_tripdata_2024-04.parquet', columns = cols)


In [19]:
df.head(10)
df.info()
df.describe()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3514289 entries, 0 to 3514288
Data columns (total 4 columns):
 #   Column                Dtype         
---  ------                -----         
 0   tpep_pickup_datetime  datetime64[us]
 1   fare_amount           float64       
 2   tip_amount            float64       
 3   payment_type          int64         
dtypes: datetime64[us](1), float64(2), int64(1)
memory usage: 107.2 MB


tpep_pickup_datetime    0
fare_amount             0
tip_amount              0
payment_type            0
dtype: int64

### Clear Data  

In [20]:
df = df[
    (df.tpep_pickup_datetime >= "2024-04-01") &
    (df.tpep_pickup_datetime <  "2024-05-01") &
    (df.payment_type == 1) &
    (df.fare_amount > 0) &
    (df.tip_amount >= 0) &
    (df.tip_amount <= 100)
]
df["tip_pct"] = df.tip_amount / df.fare_amount 
df 

,tpep_pickup_datetime,fare_amount,tip_amount,payment_type,tip_pct
0,2024-04-01 00:02:40,29.6,8.65,1,0.292230
1,2024-04-01 00:41:12,25.4,10.00,1,0.393701
2,2024-04-01 00:48:42,20.5,5.10,1,0.248780
4,2024-04-01 00:08:32,5.1,2.00,1,0.392157
6,2024-04-01 00:02:09,21.9,7.80,1,0.356164
...,...,...,...,...,...
3105707,2024-04-30 23:47:38,6.5,2.00,1,0.307692
3105708,2024-04-30 23:30:37,31.0,7.00,1,0.225806
3105709,2024-04-30 23:18:31,84.9,5.00,1,0.058893
3105710,2024-04-30 23:09:08,12.8,4.45,1,0.347656


### CI 95% 

In [21]:
sample = df.tip_amount.sample(n = 5000, random_state = 42)
n, mean = len(sample), sample.mean()
se = sample.std(ddof=1) / np.sqrt(n)
lo, hi = stats.t.interval(0.95, df=n-1, loc=mean, scale = se)
print(f"Mean = {mean:.3f}, 95% CI = [{lo:.3f}, {hi:.3f}]")

## Bootstrap CI , as right-skewed 
res = stats.bootstrap((sample.values,), np.mean, confidence_level = 0.95, 
                      n_resamples = 5000, random_state = 42)
print(f"Bootstrap CI: ", res.confidence_interval)
print("Mean:", df.tip_amount.mean())

Mean = 4.305, 95% CI = [4.195, 4.415]
Bootstrap CI:  ConfidenceInterval(low=np.float64(4.197849287065921), high=np.float64(4.419873605635952))
Mean: 4.337483794217378


### Weather Dataset


In [22]:
import requests 
url = url = ("https://archive-api.open-meteo.com/v1/archive?latitude=40.78&longitude=-73.97"
       "&start_date=2024-04-01&end_date=2024-04-30"
       "&daily=precipitation_sum&timezone=America/New_York") 
w = pd.DataFrame(requests.get(url).json()["daily"])
w["date"] = pd.to_datetime(w["time"])
w["rainy"] = w.precipitation_sum >= 1.0 

df["date"] = df.tpep_pickup_datetime.dt.normalize()
df = df.merge(w[["date", "rainy"]], on= "date", how = "left")

#### H₀: μ_mưa = μ_khô
#### H₁: μ_mưa ≠ μ_khô
#### α = 0.05

In [23]:
daily = df.groupby(["date", "rainy"]).tip_amount.mean().reset_index()
rain = daily[daily.rainy].tip_amount
dry = daily[~daily.rainy].tip_amount
print(len(rain), len(dry))

t, p = stats.ttest_ind(rain, dry, equal_var=False)
u, p_mw = stats.mannwhitneyu(rain, dry)
print(f"Welch: t={t:.3f}, p={p:.3f} | Mann-Whitney p={p_mw:.4f}")


15 15
Welch: t=-0.728, p=0.473 | Mann-Whitney p=0.2998


In [24]:
diff = rain.mean() - dry.mean()
cohen_d = diff / np.sqrt((rain.var() + dry.var()) / 2)
print(f"Chênh lệch = ${diff:.3f}, Cohen's d = {cohen_d:.2f}")

Chênh lệch = $-0.044, Cohen's d = -0.27


### Conclusion

**1. Average tip (95% CI)**
- From a random sample of 5,000 card-paid trips, the average tip is about **\$4.31**.
- We are 95% confident the true average tip is between **\$4.20 and \$4.42** (t-interval and bootstrap agree).
- The mean of all cleaned trips (**\$4.34**) falls inside this interval, so the estimate is reliable.

**2. Rainy vs. dry days (α = 0.05)**
- Rainy days tip about **\$0.04 less** than dry days (15 vs. 15 days).
- Welch t-test: p = 0.473; Mann-Whitney: p = 0.300. Both p-values are **> 0.05**, so we **fail to reject H₀**.
- The effect size is small (Cohen's d = −0.27).

**Result:** There is **no significant evidence** that rain changes the average tip in NYC taxis in April 2024.

**Limitation:** With only 30 days, the test has low power, so a small real difference could go undetected. This means "no evidence of a difference", not "proof of no difference".